# nMOS current traces

Plot `I(t)` from the PBMC nMOS history files.

The notebook looks for the three standard output directories:

- `nMOS_close`
- `nMOS_open`
- `nMOS_close_relaxed`

It accepts either the final `device_history.csv` or the live `nmos_history.csv` file.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from scipy.ndimage import gaussian_filter1d
except ImportError:
    gaussian_filter1d = None

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
})

## Locate output folders

If the notebook is run from the repository root, outputs are expected at `./nMOS_close`, etc.
If it is run from this example directory, outputs are also searched relative to the notebook directory.

In [ ]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / ".git").exists() or (path / "examples" / "PBMC" / "nMOS").exists():
            return path
    return start


cwd = Path.cwd().resolve()
repo_root = find_repo_root(cwd)
example_dir = repo_root / "examples" / "PBMC" / "nMOS"

def yaml_output_directory(config_file: Path) -> str | None:
    lines = config_file.read_text().splitlines()
    in_run_block = False
    for line in lines:
        stripped = line.strip()
        if stripped == "run:":
            in_run_block = True
            continue
        if in_run_block and line and not line.startswith(" "):
            in_run_block = False
        if in_run_block and stripped.startswith("output_directory:"):
            return stripped.split(":", 1)[1].strip().strip('"\'')
    return None


def label_from_config(config_file: Path) -> str:
    label = config_file.stem
    label = label.replace("config_nmos_", "")
    label = label.replace("config_nMOS_", "")
    label = label.replace("_", " ")
    return label


config_files = sorted(example_dir.glob("config_nmos*.yaml"))
cases = {}
for config_file in config_files:
    output_directory = yaml_output_directory(config_file)
    if output_directory:
        cases[label_from_config(config_file)] = output_directory

if not cases:
    cases = {
        "VG 0V": "nMOS_Vg_0V",
        "Vg 1V": "nMOS_Vg_1V",
        "Vg 1V relaxed": "nMOS_Vg_1V_relaxed",
    }

cases

history_filenames = ["device_history.csv", "nmos_history.csv", "simulation_history.csv"]

def find_history(case_dir_name: str) -> Path | None:
    candidate_dirs = [
        cwd / case_dir_name,
        repo_root / case_dir_name,
        example_dir / case_dir_name,
    ]
    for directory in candidate_dirs:
        for filename in history_filenames:
            path = directory / filename
            if path.exists():
                return path
    return None


history_paths = {label: find_history(directory) for label, directory in cases.items()}
history_paths

## Load histories

In [ ]:
def load_history(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [column.strip() for column in df.columns]
    if "time" not in df.columns and "time_s" in df.columns:
        df = df.rename(columns={"time_s": "time"})
    if "time" not in df.columns:
        raise ValueError(f"No time column in {path}")
    df["time_ps"] = df["time"] * 1e12
    return df


histories = {}
for label, path in history_paths.items():
    if path is None:
        print(f"missing: {label}")
        continue
    histories[label] = load_history(path)
    print(f"loaded {label}: {path} ({len(histories[label])} rows)")

if not histories:
    raise FileNotFoundError("No nMOS history files found. Run one of the YAML cases first.")

## Plot helpers

In [ ]:
def available_current_columns(df: pd.DataFrame) -> list[str]:
    preferred = [
        "probe_ramo_current",
        "ramo_current",
        "probe_ramo_current_electron",
        "probe_ramo_current_hole",
        "ramo_current_electron",
        "ramo_current_hole",
    ]
    return [column for column in preferred if column in df.columns]


def smooth_current(values, sigma: float):
    values = np.asarray(values)
    if sigma <= 0:
        return values
    if gaussian_filter1d is not None:
        return gaussian_filter1d(values, sigma)
    window = max(1, int(round(2 * sigma + 1)))
    return pd.Series(values).rolling(window=window, center=True, min_periods=1).mean().to_numpy()


current_columns = sorted({column for df in histories.values() for column in available_current_columns(df)})
current_columns

## Total and probe Ramo current

The raw signal can be noisy. Increase `Nsmooth` to make trends easier to compare.

In [ ]:
current_column = "probe_ramo_current"
tmin_ps = 0.0
Nsmooth = 5.0

fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
colors = plt.cm.jet(np.linspace(0, 1, len(histories)))
list_mean_final_current = []
list_mean_final_nbpart = []

for idx, (label, data) in enumerate(histories.items()):
    if current_column not in data.columns:
        print(f"missing column {current_column!r} for {label}")
        continue

    print(f"Processing {label}")
    time_original = data["time"].to_numpy() * 1e12
    mask = time_original > tmin_ps
    time = time_original[mask]
    ramo_current = data[current_column].to_numpy()[mask]

    axs[0].plot(time, ramo_current, label=label, color=colors[idx], linewidth=1.0)
    ramo_current_smooth = smooth_current(ramo_current, Nsmooth)
    axs[1].plot(time, ramo_current_smooth, label=label, color=colors[idx], linewidth=1.6)

    if len(ramo_current_smooth):
        tail_start = int(0.5 * len(ramo_current_smooth))
        list_mean_final_current.append((label, float(np.mean(ramo_current_smooth[tail_start:]))))
    if "nb_electrons" in data.columns:
        list_mean_final_nbpart.append((label, float(np.mean(data["nb_electrons"].to_numpy()[mask]))))

axs[0].set_ylabel("Ramo current [A]")
axs[1].set_ylabel("Ramo current [A] (smoothed)")
axs[1].set_xlabel("Time [ps]")
axs[0].set_title(f"{current_column}, raw")
axs[1].set_title(f"{current_column}, gaussian smoothing sigma={Nsmooth}")
axs[0].legend(fontsize=8)
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## Electron and hole current components

In [ ]:
component_columns = [
    "probe_ramo_current_electron",
    "probe_ramo_current_hole",
    "ramo_current_electron",
    "ramo_current_hole",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes = axes.ravel()

for axis, column in zip(axes, component_columns):
    for label, df in histories.items():
        if column not in df.columns:
            continue
        axis.plot(df["time_ps"], smooth_current(df[column], Nsmooth), label=label, linewidth=1.2)
    axis.set_title(column)
    axis.set_ylabel("Current [A]")
    axis.legend()

for axis in axes[-2:]:
    axis.set_xlabel("Time [ps]")

fig.tight_layout()
plt.show()

## Particle counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for label, df in histories.items():
    if "nb_electrons" in df.columns:
        axes[0].plot(df["time_ps"], df["nb_electrons"], label=label)
    if "nb_holes" in df.columns:
        axes[1].plot(df["time_ps"], df["nb_holes"], label=label)

axes[0].set_title("Electrons")
axes[1].set_title("Holes")
for axis in axes:
    axis.set_xlabel("Time [ps]")
    axis.set_ylabel("Numerical particles")
    axis.legend()

fig.tight_layout()
plt.show()

## Steady-state averages

The cell below discards the first `transient_fraction` of each trace and reports mean/std current values.

In [ ]:
transient_fraction = 0.5

rows = []
for label, df in histories.items():
    start_index = int(len(df) * transient_fraction)
    steady = df.iloc[start_index:]
    row = {"case": label, "rows": len(df), "steady_rows": len(steady)}
    for column in ["probe_ramo_current", "ramo_current"]:
        if column in steady.columns and not steady.empty:
            row[f"{column}_mean_A"] = steady[column].mean()
            row[f"{column}_std_A"] = steady[column].std()
    rows.append(row)

summary = pd.DataFrame(rows)
summary

## Save figures

Set `save_plots = True` and rerun plotting cells if you want persistent PNGs.

In [ ]:
save_plots = False
plot_dir = example_dir / "figs"

if save_plots:
    plot_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(plot_dir / "current_summary.csv", index=False)
    print(f"Saved summary to {plot_dir / 'current_summary.csv'}")